# Pipeline Orquestador — Embalse Guatapé

Orquesta 11 steps en SageMaker:
1. Ingesta
2. Procesamiento
3. Entrenamiento ARIMA → 4. Inferencia ARIMA
5. Entrenamiento GARCH → 6. Inferencia GARCH
7. Entrenamiento HW    → 8. Inferencia HW
9. Entrenamiento LSTM  → 10. Inferencia LSTM
11. Dashboard

Los steps 3-10 corren en paralelo después del procesamiento.
El dashboard espera a que terminen todas las inferencias.

---
**Solo correr completo cuando se modifique la estructura del pipeline.**  
Para ejecución manual: correr celdas 1-2 y luego directamente 'Lanzar ejecución'.

In [1]:
import boto3
import json
import sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.pipeline_context import PipelineSession
print('OK')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
OK


In [2]:
# ── Configuración central ─────────────────────────────────────────────────────
BUCKET        = 'embalses-colombia'
ROLE          = 'arn:aws:iam::025627370678:role/SageMakerExecutionRole'
REGION        = 'us-east-1'
PIPELINE_NAME = 'pipeline-guatape'
IMAGE_URI     = '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-cpu-py3'
INSTANCE_TYPE = 'ml.t3.medium'

SCRIPTS   = f's3://{BUCKET}/scripts'
CURATED   = f's3://{BUCKET}/data/curated/embalse_guatape/'

# Modelos
MODEL_ARIMA = f's3://{BUCKET}/models/arima/latest/'
MODEL_GARCH = f's3://{BUCKET}/models/garch/latest/'
MODEL_HW    = f's3://{BUCKET}/models/hw/latest/'
MODEL_LSTM  = f's3://{BUCKET}/models/lstm/latest/'

# Predicciones
PREDS_ARIMA = f's3://{BUCKET}/predictions/arima/'
PREDS_GARCH = f's3://{BUCKET}/predictions/garch/'
PREDS_HW    = f's3://{BUCKET}/predictions/hw/'
PREDS_LSTM  = f's3://{BUCKET}/predictions/lstm/'

session = PipelineSession(boto_session=boto3.Session(region_name=REGION))
print('Configuración OK')

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


Configuración OK


In [3]:
# ── Procesador base ───────────────────────────────────────────────────────────
def make_processor():
    return ScriptProcessor(
        image_uri=IMAGE_URI,
        command=['python3'],
        instance_type=INSTANCE_TYPE,
        instance_count=1,
        role=ROLE,
        sagemaker_session=session,
    )

In [4]:
# ── Step 1: Ingesta ───────────────────────────────────────────────────────────
step_ingesta = ProcessingStep(
    name='ingesta-guatape',
    processor=make_processor(),
    code=f'{SCRIPTS}/ingesta_guatape.py',
)

In [5]:
# ── Step 2: Procesamiento ─────────────────────────────────────────────────────
step_procesamiento = ProcessingStep(
    name='procesamiento-guatape',
    processor=make_processor(),
    code=f'{SCRIPTS}/procesamiento_guatape.py',
    depends_on=[step_ingesta],
)

In [6]:
# ── Steps 3-4: ARIMA ─────────────────────────────────────────────────────────
step_train_arima = ProcessingStep(
    name='entrenamiento-arima',
    processor=make_processor(),
    code=f'{SCRIPTS}/entrenamiento_arima.py',
    inputs=[ProcessingInput(source=CURATED, destination='/opt/ml/processing/input/curated/embalse_guatape/')],
    outputs=[ProcessingOutput(source='/opt/ml/processing/output/model', destination=MODEL_ARIMA, output_name='model-arima')],
    depends_on=[step_procesamiento],
)

step_infer_arima = ProcessingStep(
    name='inferencia-arima',
    processor=make_processor(),
    code=f'{SCRIPTS}/inferencia_arima.py',
    inputs=[ProcessingInput(source=MODEL_ARIMA, destination='/opt/ml/processing/input/model')],
    outputs=[ProcessingOutput(source='/opt/ml/processing/output/preds', destination=PREDS_ARIMA, output_name='preds-arima')],
    depends_on=[step_train_arima],
)

In [7]:
# ── Steps 5-6: GARCH ─────────────────────────────────────────────────────────
step_train_garch = ProcessingStep(
    name='entrenamiento-garch',
    processor=make_processor(),
    code=f'{SCRIPTS}/entrenamiento_garch.py',
    inputs=[ProcessingInput(source=CURATED, destination='/opt/ml/processing/input/curated/embalse_guatape/')],
    outputs=[ProcessingOutput(source='/opt/ml/processing/output/model', destination=MODEL_GARCH, output_name='model-garch')],
    depends_on=[step_procesamiento],
)

step_infer_garch = ProcessingStep(
    name='inferencia-garch',
    processor=make_processor(),
    code=f'{SCRIPTS}/inferencia_garch.py',
    inputs=[
        ProcessingInput(source=MODEL_GARCH, destination='/opt/ml/processing/input/model'),
        ProcessingInput(source=CURATED, destination='/opt/ml/processing/input/curated'),
    ],
    outputs=[ProcessingOutput(source='/opt/ml/processing/output/preds', destination=PREDS_GARCH, output_name='preds-garch')],
    depends_on=[step_train_garch],
)

In [8]:
# ── Steps 7-8: Holt-Winters ───────────────────────────────────────────────────
step_train_hw = ProcessingStep(
    name='entrenamiento-hw',
    processor=make_processor(),
    code=f'{SCRIPTS}/entrenamiento_hw.py',
    inputs=[ProcessingInput(source=CURATED, destination='/opt/ml/processing/input/curated/embalse_guatape/')],
    outputs=[ProcessingOutput(source='/opt/ml/processing/output/model', destination=MODEL_HW, output_name='model-hw')],
    depends_on=[step_procesamiento],
)

step_infer_hw = ProcessingStep(
    name='inferencia-hw',
    processor=make_processor(),
    code=f'{SCRIPTS}/inferencia_hw.py',
    inputs=[ProcessingInput(source=MODEL_HW, destination='/opt/ml/processing/input/model')],
    outputs=[ProcessingOutput(source='/opt/ml/processing/output/preds', destination=PREDS_HW, output_name='preds-hw')],
    depends_on=[step_train_hw],
)

In [9]:
# ── Steps 9-10: LSTM ─────────────────────────────────────────────────────────
step_train_lstm = ProcessingStep(
    name='entrenamiento-lstm',
    processor=make_processor(),
    code=f'{SCRIPTS}/entrenamiento_lstm.py',
    inputs=[ProcessingInput(source=CURATED, destination='/opt/ml/processing/input/curated/embalse_guatape/')],
    outputs=[ProcessingOutput(source='/opt/ml/processing/output/model', destination=MODEL_LSTM, output_name='model-lstm')],
    depends_on=[step_procesamiento],
)

step_infer_lstm = ProcessingStep(
    name='inferencia-lstm',
    processor=make_processor(),
    code=f'{SCRIPTS}/inferencia_lstm.py',
    inputs=[ProcessingInput(source=MODEL_LSTM, destination='/opt/ml/processing/input/model')],
    outputs=[ProcessingOutput(source='/opt/ml/processing/output/preds', destination=PREDS_LSTM, output_name='preds-lstm')],
    depends_on=[step_train_lstm],
)

In [10]:
# ── Step 11: Dashboard ────────────────────────────────────────────────────────
step_dashboard = ProcessingStep(
    name='dashboard-guatape',
    processor=make_processor(),
    code=f'{SCRIPTS}/dashboard.py',
    depends_on=[step_infer_arima, step_infer_garch, step_infer_hw, step_infer_lstm],
)

In [11]:
# ── Registrar pipeline ────────────────────────────────────────────────────────
pipeline = Pipeline(
    name=PIPELINE_NAME,
    steps=[
        step_ingesta,
        step_procesamiento,
        step_train_arima, step_infer_arima,
        step_train_garch, step_infer_garch,
        step_train_hw,    step_infer_hw,
        step_train_lstm,  step_infer_lstm,
        step_dashboard,
    ],
    sagemaker_session=session,
)
pipeline.upsert(role_arn=ROLE)
print(f'Pipeline "{PIPELINE_NAME}" registrado en SageMaker.')

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline "pipeline-guatape" registrado en SageMaker.


In [18]:
# ── Lanzar ejecución manual ───────────────────────────────────────────────────
execution = pipeline.start()
print(f'Ejecución iniciada: {execution.arn}')

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Ejecución iniciada: arn:aws:sagemaker:us-east-1:025627370678:pipeline/pipeline-guatape/execution/vn5xihkidflc


In [41]:
# ── Monitorear estado ─────────────────────────────────────────────────────────
import boto3
sm = boto3.client('sagemaker', region_name=REGION)
latest = sm.list_pipeline_executions(PipelineName=PIPELINE_NAME, SortOrder='Descending')['PipelineExecutionSummaries'][0]
print(f'Estado: {latest["PipelineExecutionStatus"]}')
steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=latest['PipelineExecutionArn'])
for s in steps['PipelineExecutionSteps']:
    print(f"  {s['StepName']:30s} → {s['StepStatus']}")

Estado: Failed
  dashboard-guatape              → Failed
  inferencia-lstm                → Succeeded
  inferencia-garch               → Succeeded
  inferencia-arima               → Succeeded
  inferencia-hw                  → Succeeded
  entrenamiento-hw               → Succeeded
  entrenamiento-lstm             → Succeeded
  entrenamiento-arima            → Succeeded
  entrenamiento-garch            → Succeeded
  procesamiento-guatape          → Succeeded
  ingesta-guatape                → Succeeded


In [ ]:
# ── EventBridge: ejecución automática diaria ──────────────────────────────────
# Horario actual: 12pm Colombia (17:00 UTC)
# Para cambiar la hora: cron(MINUTO HORA_UTC * * ? *)
# Solo ejecutar si se necesita cambiar el horario

events = boto3.client('events', region_name=REGION)
PIPELINE_ARN = f'arn:aws:sagemaker:{REGION}:025627370678:pipeline/{PIPELINE_NAME}'

events.put_rule(
    Name='pipeline-guatape-diario',
    ScheduleExpression='cron(0 17 * * ? *)',
    State='ENABLED',
    Description='Ejecución diaria del pipeline Guatapé a las 12pm Colombia',
)
events.put_targets(
    Rule='pipeline-guatape-diario',
    Targets=[{'Id': 'pipeline-guatape-target', 'Arn': PIPELINE_ARN, 'RoleArn': ROLE}]
)
print('EventBridge configurado OK')

In [ ]:
# ── SNS: notificaciones por email ─────────────────────────────────────────────
# Para agregar otro destinatario: cambiar EMAIL y ejecutar
# Solo ejecutar si se necesita cambiar el email

TOPIC_ARN = 'arn:aws:sns:us-east-1:025627370678:pipeline-guatape-notificaciones'
EMAIL     = 'jdtrujillo1015@hotmail.com'
PIPELINE_ARN = f'arn:aws:sagemaker:{REGION}:025627370678:pipeline/{PIPELINE_NAME}'

sns = boto3.client('sns', region_name=REGION)
sns.subscribe(TopicArn=TOPIC_ARN, Protocol='email', Endpoint=EMAIL)

sns.set_topic_attributes(
    TopicArn=TOPIC_ARN,
    AttributeName='Policy',
    AttributeValue=json.dumps({
        'Version': '2012-10-17',
        'Statement': [{'Effect': 'Allow', 'Principal': {'Service': 'events.amazonaws.com'},
                       'Action': 'sns:Publish', 'Resource': TOPIC_ARN}]
    })
)

events = boto3.client('events', region_name=REGION)
events.put_rule(
    Name='pipeline-guatape-estado',
    EventPattern=json.dumps({
        'source': ['aws.sagemaker'],
        'detail-type': ['SageMaker Model Building Pipeline Execution Status Change'],
        'detail': {'pipelineArn': [PIPELINE_ARN],
                   'currentPipelineExecutionStatus': ['Succeeded', 'Failed']}
    }),
    State='ENABLED',
    Description='Notificación cuando el pipeline Guatapé termina o falla',
)
events.put_targets(Rule='pipeline-guatape-estado', Targets=[{'Id': 'notificacion-sns', 'Arn': TOPIC_ARN}])
print(f'SNS configurado — confirma el email en {EMAIL}')